In [3]:
from wellness_tracker import run_select

def get_view_combo_names():
    """
    This function retrieves the combo names of the views in order to add to the food dropdown widget. It queries the pg_views table to get 
    the names of all views in the public schema, then constructs a SQL query that unions the distinct combo_names from each view, and 
    finally returns a list of all combo names.
    """
    query = """
        SELECT viewname
        FROM pg_views
        WHERE schemaname = 'public'
        ORDER BY schemaname, viewname;
    """
    views = run_select(query, return_df=True)['viewname'].tolist()
    
    # create one SQL query that unions all the distinct combo_names from each view
    union_sql = " UNION ".join(
    f"SELECT DISTINCT combo_name AS name FROM {view}"
    for view in views
    )
    
    df = run_select(union_sql, return_df=True)
    all_combo_names = df['name'].tolist()
    return all_combo_names

    
get_view_combo_names()    

['-YGRT_DT_NT', '-YGRT_PB_FRT_NT', '-YGRT_RSN_NT']

## will need to update the food insert function as our views have different columns that the function assumes.

## we are also removing the combo_id column.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML, Markdown
from AI import OpenAIClient
from postgres import run_select
from wellness_tracker import WellnessTracker

# Build widget data
all_foods = []
views = (get_view_combo_names())
all_foods.extend(views)

reg_foods = run_select(
    """
    select name from food
    order by name;
    """,
    return_df=True,
)['name'].tolist()

all_foods.extend(reg_foods)

foods = all_foods

combo_name = widgets.Text(
    value="",
    description="Combo Name:",
    placeholder="Enter name"
)


food_search = widgets.Text(
    description="Search:",
    placeholder="Type to filter foods",
)

food = widgets.Dropdown(
    options=all_foods,
    description="Food:",
)


def _sync_food_options(*_):
    query = (food_search.value or "").strip().lower()
    filtered_foods = [item for item in foods if query in str(item).lower()]
    food.options = filtered_foods
    food.value = filtered_foods[0] if filtered_foods else None


food_search.observe(_sync_food_options, names="value")
_sync_food_options()

serving_amt = widgets.Dropdown(
    # only show decimal options that are multiples of 0.5, up to 10, using floor division and modulus to determine if it's a whole number or half
    options=[i // 2 if i % 2 == 0 else i / 2 for i in range(1, 21)],
    description="Quantity:", value=1.0,
)

add_food = widgets.Button(description="Add Food")
out = widgets.Output()

food_list = []
serving_size_list = []

def on_add_food_clicked(_):
    food_list.append(food.value)
    serving_size_list.append(serving_amt.value)
    if len(food_list) > 10:
        print("Maximum of 10 foods allowed.")
        food_list.pop()
    if len(serving_size_list) > 10:
        print("Maximum of 10 foods allowed.")
        serving_size_list.pop()   

remove_last_food = widgets.Button(description="Remove Last Food")
def on_remove_last_food_clicked(_):
    if len(food_list) > 0:
        food_list.pop()
        if len(serving_size_list) == len(food_list)+1:
           serving_size_list.pop()  
           
add_food.on_click(on_add_food_clicked)
remove_last_food.on_click(on_remove_last_food_clicked)

wt = WellnessTracker()

  
    